In [25]:
import pandas as pd

In [26]:
# LOAD DATA
df_edu = pd.read_csv('../data/processed/ci_education_15_35.csv')
df_sector = pd.read_csv('../data/processed/ci_sector_15_35.csv')

In [27]:
# VULNÉRABILITÉ ÉDUCATION 
df_edu['vulnerability'] = (df_edu['share_unemp'] + df_edu['share_inact']) / 2

edu_summary = df_edu.groupby('edu_ilo')['vulnerability'].mean().reset_index()

print("\n=== VULNÉRABILITÉ PAR NIVEAU D'ÉDUCATION ===")
print(edu_summary)


=== VULNÉRABILITÉ PAR NIVEAU D'ÉDUCATION ===
             edu_ilo  vulnerability
0  Less than primary       0.154792
1    Lower secondary       0.255324
2            Primary       0.222519
3           Tertiary       0.312390
4    Upper secondary       0.299516


In [28]:

# STRUCTURE EMPLOI PAR SECTEUR

sector_summary = df_sector.groupby('sector21')['population'].sum().reset_index()

sector_summary['share'] = sector_summary['population'] / sector_summary['population'].sum()

print("\n=== TOP SECTEURS D'EMPLOI ===")
print(sector_summary.sort_values('share', ascending=False).head(10))


=== TOP SECTEURS D'EMPLOI ===
                                             sector21  population     share
4                   Agriculture; forestry and fishing  28897468.0  0.486310
20  Wholesale and retail trade; repair of motor ve...  11249038.0  0.189308
12                                      Manufacturing   4804207.0  0.080849
0           Accommodation and food service activities   2692696.0  0.045315
14                           Other service activities   2369389.0  0.039874
18                         Transportation and storage   2016180.0  0.033930
6                                        Construction   1650030.0  0.027768
2   Activities of households as employers; undiffe...   1244720.0  0.020947
7                                           Education    910465.0  0.015322
16  Public administration and defence; compulsory ...    834494.0  0.014044


In [29]:
# 4. SCORE GLOBAL DE DÉSALIGNEMENT

global_vulnerability = df_edu['vulnerability'].mean()

sector_summary['risk_contribution'] = sector_summary['share'] * global_vulnerability

sector_summary = sector_summary.sort_values('risk_contribution', ascending=False)

print("\n=== SCORE D'IMPACT PAR SECTEUR ===")
print(sector_summary.head(10))


=== SCORE D'IMPACT PAR SECTEUR ===
                                             sector21  population     share  \
4                   Agriculture; forestry and fishing  28897468.0  0.486310   
20  Wholesale and retail trade; repair of motor ve...  11249038.0  0.189308   
12                                      Manufacturing   4804207.0  0.080849   
0           Accommodation and food service activities   2692696.0  0.045315   
14                           Other service activities   2369389.0  0.039874   
18                         Transportation and storage   2016180.0  0.033930   
6                                        Construction   1650030.0  0.027768   
2   Activities of households as employers; undiffe...   1244720.0  0.020947   
7                                           Education    910465.0  0.015322   
16  Public administration and defence; compulsory ...    834494.0  0.014044   

    risk_contribution  
4            0.121047  
20           0.047120  
12           0.020124 

In [30]:
# INDICATEUR FINAL

agriculture_share = sector_summary[sector_summary['sector21'].str.contains('Agriculture', case=False)]['share'].sum()
commerce_share = sector_summary[sector_summary['sector21'].str.contains('trade|retail|commerce', case=False)]['share'].sum()

total_exposure = agriculture_share + commerce_share

print("\n=== INSIGHT FINAL ===")
print(f"Exposition agriculture + commerce : {total_exposure:.2f}")

if total_exposure > 0.6:
    print(" Forte concentration des jeunes dans secteurs vulnérables")
else:
    print(" Structure plus diversifiée")


=== INSIGHT FINAL ===
Exposition agriculture + commerce : 0.68
 Forte concentration des jeunes dans secteurs vulnérables


In [31]:
# VULNÉRABILITÉ GLOBALE 

df_edu['vulnerability'] = (df_edu['share_unemp'] + df_edu['share_inact']) / 2

global_vuln = df_edu['vulnerability'].mean()

print("\n=== VULNÉRABILITÉ GLOBALE JEUNES ===")
print(round(global_vuln, 4))


=== VULNÉRABILITÉ GLOBALE JEUNES ===
0.2489


In [32]:
# STRUCTURE DE L’EMPLOI PAR SECTEUR

sector_summary = df_sector.groupby('sector21')['population'].sum().reset_index()

sector_summary['share'] = sector_summary['population'] / sector_summary['population'].sum()

print("\n=== STRUCTURE EMPLOI PAR SECTEUR ===")
print(sector_summary.sort_values('share', ascending=False).head(10))


=== STRUCTURE EMPLOI PAR SECTEUR ===
                                             sector21  population     share
4                   Agriculture; forestry and fishing  28897468.0  0.486310
20  Wholesale and retail trade; repair of motor ve...  11249038.0  0.189308
12                                      Manufacturing   4804207.0  0.080849
0           Accommodation and food service activities   2692696.0  0.045315
14                           Other service activities   2369389.0  0.039874
18                         Transportation and storage   2016180.0  0.033930
6                                        Construction   1650030.0  0.027768
2   Activities of households as employers; undiffe...   1244720.0  0.020947
7                                           Education    910465.0  0.015322
16  Public administration and defence; compulsory ...    834494.0  0.014044


In [33]:
# SCORE DE QUALITÉ / RISQUE SECTORIEL

sector_summary['risk_score'] = sector_summary['share'] * global_vuln

sector_summary = sector_summary.sort_values('risk_score', ascending=False)

print("\n=== TOP SECTEURS À IMPACT VULNÉRABILITÉ ===")
print(sector_summary.head(10))


=== TOP SECTEURS À IMPACT VULNÉRABILITÉ ===
                                             sector21  population     share  \
4                   Agriculture; forestry and fishing  28897468.0  0.486310   
20  Wholesale and retail trade; repair of motor ve...  11249038.0  0.189308   
12                                      Manufacturing   4804207.0  0.080849   
0           Accommodation and food service activities   2692696.0  0.045315   
14                           Other service activities   2369389.0  0.039874   
18                         Transportation and storage   2016180.0  0.033930   
6                                        Construction   1650030.0  0.027768   
2   Activities of households as employers; undiffe...   1244720.0  0.020947   
7                                           Education    910465.0  0.015322   
16  Public administration and defence; compulsory ...    834494.0  0.014044   

    risk_score  
4     0.121047  
20    0.047120  
12    0.020124  
0     0.011279  


In [34]:
# INTERPRÉTATION AUTOMATIQUE (INSIGHT CLÉ)

agri = sector_summary[sector_summary['sector21'].str.contains('Agriculture', case=False)]['share'].sum()
commerce = sector_summary[sector_summary['sector21'].str.contains('trade|retail|commerce', case=False)]['share'].sum()

total_risk_concentration = agri + commerce

print("\n=== INSIGHT STRUCTUREL ===")
print(f"Concentration Agriculture + Commerce : {round(total_risk_concentration, 2)}")

if total_risk_concentration > 0.6:
    print(" Forte dépendance aux secteurs à faible qualité d’emploi")
else:
    print(" Structure plus diversifiée du marché du travail")


=== INSIGHT STRUCTUREL ===
Concentration Agriculture + Commerce : 0.68
 Forte dépendance aux secteurs à faible qualité d’emploi


In [35]:

# STRUCTURE DE L'ÉDUCATION (OFFRE DE COMPÉTENCES)

edu_dist = df_edu.groupby('edu_ilo')['share_unemp'].mean().reset_index()
edu_dist = edu_dist.rename(columns={'share_unemp': 'avg_unemployment'})

# normalisation simple pour comparaison
edu_dist['edu_share_proxy'] = edu_dist['avg_unemployment'] / edu_dist['avg_unemployment'].sum()

In [36]:
# 3. STRUCTURE DE L'EMPLOI (DEMANDE DU MARCHÉ)

sector_dist = df_sector.groupby('sector21')['population'].sum().reset_index()
sector_dist['job_share'] = sector_dist['population'] / sector_dist['population'].sum()

In [37]:
# CRÉATION D'UN MISMATCH INDEX
# idée: plus les distributions sont différentes, plus le mismatch est élevé

# on prend top secteurs et on compare avec niveau global
top_sectors = sector_dist.sort_values('job_share', ascending=False).head(5)

mismatch_score = abs(edu_dist['edu_share_proxy'].sum() - top_sectors['job_share'].sum())

print("\n=== MISMATCH GLOBAL ÉDUCATION - EMPLOI ===")
print(f"Score de mismatch : {round(mismatch_score, 4)}")


=== MISMATCH GLOBAL ÉDUCATION - EMPLOI ===
Score de mismatch : 0.1583


In [38]:
# INSIGHT STRUCTUREL

agriculture_share = sector_dist[sector_dist['sector21'].str.contains('Agriculture', case=False)]['job_share'].sum()
commerce_share = sector_dist[sector_dist['sector21'].str.contains('trade|retail|wholesale', case=False)]['job_share'].sum()

total_concentration = agriculture_share + commerce_share

print("\n=== STRUCTURE DU DÉCALAGE ===")
print(f"Agriculture + Commerce = {round(total_concentration, 2)}")

if total_concentration > 0.6:
    print(" Forte économie de subsistance → mismatch structurel élevé")
else:
    print(" Structure économique relativement diversifiée")




=== STRUCTURE DU DÉCALAGE ===
Agriculture + Commerce = 0.68
 Forte économie de subsistance → mismatch structurel élevé


In [39]:
# CONCLUSION 

print("\n=== CONCLUSION ===")

if mismatch_score > 0.3:
    print(" Désalignement critique entre formation et emploi")
    print(" Le marché absorbe mal les compétences disponibles")
else:
    print(" Alignement partiel entre éducation et emploi")


=== CONCLUSION ===
 Alignement partiel entre éducation et emploi
